In [ ]:
"""
Universal Multiplex Neural Manifold Engine 
Features: Cross-Method Fusion, Pure MEG_All_Bands Support, Z-Score Shield, CSV Export
"""

import os
import glob
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore
from collections import defaultdict
import itertools
import gc
import warnings

warnings.filterwarnings('ignore')

# ==========================================================
# 1. Configuration & Directories
# ==========================================================
FMRI_DIR = r"E:\Results\fmri_results"
MEG_DIR = r"E:\Results\meg_results"
OUTPUT_DIR = r"E:\Results\Manifold_Plots_Multiplex"

METHODS = ['Pearson', 'GCA', 'mTE']

MATRIX_KEYS = {
    "Pearson": "results/results/corr_matrix",
    "GCA": "results/results/gca_fdr_pval",
    "mTE": "results/te_weighted_adjacency_matrix"
}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================================
# 2. Data Loading & Parsing
# ==========================================================
def extract_matrix(h5_path, method):
    matrix_key = MATRIX_KEYS.get(method)
    try:
        with h5py.File(h5_path, 'r') as hf:
            if matrix_key in hf:
                mat = hf[matrix_key][:]
                if method == 'GCA':
                    mat = np.nan_to_num(mat, nan=1.0)
                    mat = -np.log10(np.clip(mat, 1e-15, 1.0))
                upper_tri = mat[np.triu_indices_from(mat, k=1)]
                return np.nan_to_num(upper_tri, nan=0.0)
    except Exception as e:
        print(f"[Warning] Failed reading {os.path.basename(h5_path)}: {e}")
    return None

def build_master_database():
    db = {meth: defaultdict(lambda: defaultdict(list)) for meth in METHODS}
    
    if os.path.exists(FMRI_DIR):
        for fpath in glob.glob(os.path.join(FMRI_DIR, "*.h5")):
            parts = os.path.basename(fpath).replace('.h5', '').split('_')
            if len(parts) >= 3 and parts[0] in METHODS:
                meth, subj = parts[0], parts[1]
                vec = extract_matrix(fpath, meth)
                if vec is not None:
                    db[meth]["fMRI"][subj].append(vec)
                    
    if os.path.exists(MEG_DIR):
        for fpath in glob.glob(os.path.join(MEG_DIR, "*.h5")):
            parts = os.path.basename(fpath).replace('.h5', '').split('_')
            if len(parts) >= 3 and parts[0] in METHODS:
                meth, subj = parts[0], parts[1]
                task_str = "_".join(parts[2:])
                band = "Wide-band"
                for b in ["Alpha", "Beta", "Delta", "Theta", "Low-Gamma", "High-Gamma", "Low-Frequency", "Wide-band", "Aperiodic-VMD", "Aperiodic"]:
                    if b in task_str:
                        band = b if b != "Aperiodic" else "Aperiodic-VMD"
                        break
                vec = extract_matrix(fpath, meth)
                if vec is not None:
                    db[meth][f"MEG_{band}"][subj].append(vec)
    return db

# ==========================================================
# 3. Dynamic Fusion, Projection & Export
# ==========================================================
def process_and_export_manifold(features, subjects, title_name):
    print(f"-> Projecting Multiplex Manifold: {title_name} (N={len(subjects)} samples)...")
    
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    features_scaled = np.nan_to_num(features_scaled, nan=0.0)
    
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, metric='cosine', random_state=42)
    embedding = reducer.fit_transform(features_scaled)
    
    df_plot = pd.DataFrame({
        'Subject_ID': subjects,
        'UMAP_Dimension_1': embedding[:, 0],
        'UMAP_Dimension_2': embedding[:, 1],
        'Analysis_Group': title_name
    })
    
    safe_name = title_name.replace(" + ", "_Plus_").replace(" ", "_").replace("[", "").replace("]", "")
    
    # Export CSV
    df_plot.to_csv(os.path.join(OUTPUT_DIR, f"Manifold_{safe_name}.csv"), index=False)
    
    # Export PNG
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x='UMAP_Dimension_1', y='UMAP_Dimension_2',
        hue='Subject_ID', palette='tab20', data=df_plot,
        s=90, alpha=0.8, edgecolor='k', legend=False
    )
    plt.title(f"Multiplex Neural Manifold:\n{title_name}", fontsize=14, fontweight='bold')
    plt.xlabel("UMAP Dimension 1", fontsize=12)
    plt.ylabel("UMAP Dimension 2", fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"Manifold_{safe_name}.png"), dpi=300)
    plt.close()

def execute_multiplex_pipeline(db):
    single_mods = set()
    for meth in METHODS:
        single_mods.update(db[meth].keys())
        
    modality_targets = {mod: [mod] for mod in single_mods}
    
    if "fMRI" in single_mods and "MEG_Aperiodic-VMD" in single_mods:
        modality_targets["fMRI + MEG_Aperiodic"] = ["fMRI", "MEG_Aperiodic-VMD"]
        
    meg_bands = [m for m in single_mods if m.startswith("MEG_")]
    
    # =========================================================
    # THE ABLATION ADDITIONS: Pure MEG vs Ultimate Fusion
    # =========================================================
    if len(meg_bands) > 1:
        modality_targets["MEG_All_Bands"] = meg_bands
        
    if "fMRI" in single_mods and len(meg_bands) > 1:
        modality_targets["fMRI + MEG_All_Bands"] = ["fMRI"] + meg_bands

    method_combos = []
    for r in range(1, len(METHODS) + 1):
        method_combos.extend(list(itertools.combinations(METHODS, r)))

    for mod_name, req_mods in modality_targets.items():
        for meth_combo in method_combos:
            combo_name = "+".join(meth_combo)
            title_name = f"{combo_name} [{mod_name}]"
            
            valid_subjs = None
            is_valid = True
            
            for meth in meth_combo:
                for mod in req_mods:
                    if mod not in db[meth]:
                        is_valid = False
                        break
                    subjs = set(s for s, runs in db[meth][mod].items() if len(runs) >= 2)
                    if valid_subjs is None:
                        valid_subjs = subjs
                    else:
                        valid_subjs = valid_subjs.intersection(subjs)
                        
            if not is_valid or valid_subjs is None or len(valid_subjs) < 5:
                continue
                
            features, labels = [], []
            for s in sorted(valid_subjs):
                min_runs = min(len(db[meth][mod][s]) for meth in meth_combo for mod in req_mods)
                
                for run_idx in range(min_runs):
                    pieces = []
                    for meth in meth_combo:
                        for mod in req_mods:
                            vec = db[meth][mod][s][run_idx]
                            
                            z_vec = zscore(vec)
                            z_vec_safe = np.nan_to_num(z_vec, nan=0.0)
                            pieces.append(z_vec_safe)
                            
                    fused_vec = np.concatenate(pieces)
                    features.append(fused_vec)
                    labels.append(s)
                    
            process_and_export_manifold(np.array(features), np.array(labels), title_name)
            
            del features, labels
            gc.collect()

# ==========================================================
# 4. Main Execution
# ==========================================================
if __name__ == "__main__":
    print("="*70)
    print(" Initiating Universal Multiplex Neural Manifold Engine ")
    print("   [Feature: Full Ablation Support & Bulletproof NaN Shield]")
    print("="*70)
    
    master_db = build_master_database()
    execute_multiplex_pipeline(master_db)
    
    print("="*70)
    print(" ALL MULTIPLEX MANIFOLD PLOTS AND CSVs GENERATED SUCCESSFULLY!")
    print(f"Check directory: {OUTPUT_DIR}")
    print("="*70)